# Training + Monitoring IMDB BiLSTM
**Dicoding - Membangun Sistem Machine Learning**

Notebook ini akan:
1. Clone repo dari GitHub
2. Install dependencies
3. Jalankan training + tuning
4. Serve model + monitoring

In [ ]:
# @title Clone repo dari GitHub
import os

GITHUB_REPO = "https://github.com/Dianftnh/Dicoding-Training_Model.git"
PROJECT_DIR = "/content/Dicoding-Training_Model"

if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
print(f"Working dir: {os.getcwd()}")

In [ ]:
# @title Install dependencies
!pip install -q tensorflow mlflow scikit-learn numpy matplotlib pandas requests prometheus_client

---
## 🔥 Training Model

In [ ]:
# @title Jalankan Basic Training (modelling.py)
%cd {PROJECT_DIR}/Membangun_model
!python modelling.py

In [ ]:
# @title Jalankan Hyperparameter Tuning (modelling_tuning.py)
%cd {PROJECT_DIR}/Membangun_model
!python modelling_tuning.py

In [ ]:
# @title Lihat hasil MLflow
!mlflow ui --host 0.0.0.0 --port 5000 &
import time
time.sleep(3)

# Install ngrok untuk akses MLflow UI
!pip install -q pyngrok
from pyngrok import ngrok

NGROK_TOKEN = "" # @param {type:"string"}
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    url = ngrok.connect(5000)
    print(f"MLflow UI: {url}")
else:
    print("MLflow UI berjalan di localhost:5000")
    print("Gunakan ngrok token untuk akses publik")

### 📸 Screenshot
- **Dashboard**: Halaman utama MLflow (daftar runs + metrics)
- **Artifacts**: Klik salah satu run → tab Artifacts

Simpan sebagai `screenshoot_dashboard.png` dan `screenshoot_artifak.png`

---
## 🚀 Model Serving

In [ ]:
# @title Cari path model artifact terbaru
import glob
import os

model_dirs = sorted(glob.glob(f"{PROJECT_DIR}/Membangun_model/mlruns/*/models/*/artifacts/MLmodel"))
for p in model_dirs:
    print(p)

if model_dirs:
    ARTIFACTS_DIR = os.path.dirname(model_dirs[-1])
    print(f"\nModel terbaru: {ARTIFACTS_DIR}")
else:
    print("Tidak ada model!")

In [ ]:
# @title Start MLflow serving
import subprocess, time, requests

log = open('/content/mlflow_serve.log', 'w')
serve_proc = subprocess.Popen(
    ['mlflow', 'models', 'serve', '-m', ARTIFACTS_DIR, '--port', '8080', '--no-conda'],
    stdout=log, stderr=log
)
print(f"PID: {serve_proc.pid}")

for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8080/ping", timeout=2)
        if r.status_code == 200:
            print(f"Server siap ({i+1}s)")
            break
    except:
        pass
    time.sleep(1)
else:
    print("Server gagal start")
    !cat /content/mlflow_serve.log

In [ ]:
# @title Uji Inference
import requests, json, numpy as np, pickle

DATA_DIR = f"{PROJECT_DIR}/Membangun_model/dataset_preprocessing"
X_test = np.load(f"{DATA_DIR}/X_test.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
with open(f"{DATA_DIR}/label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

sample = X_test[:5]
r = requests.post(
    "http://127.0.0.1:8080/invocations",
    json={"instances": sample.tolist()}
)

print(f"Status: {r.status_code}")
if r.status_code == 200:
    preds = np.argmax(r.json()["predictions"], axis=1)
    for i in range(5):
        p = encoder.inverse_transform([preds[i]])[0]
        a = encoder.inverse_transform([y_test[i]])[0]
        print(f"{i}: Pred={p:>7} | Actual={a:>7} {'✅' if preds[i]==y_test[i] else '❌'}")

### 📸 Screenshot hasil inference → `1.bukti_serving/bukti_serving.png`

---
## 📊 Monitoring

In [ ]:
# @title Jalankan Prometheus Exporter
exporter_log = open('/content/exporter.log', 'w')
exporter_proc = subprocess.Popen(
    ['python', f'{PROJECT_DIR}/Monitoring_dan_Logging/3.prometheus_exporter.py'],
    stdout=exporter_log, stderr=exporter_log
)
print(f"Exporter PID: {exporter_proc.pid}")

time.sleep(10)
r = requests.get("http://127.0.0.1:8001/metrics")
print(r.text[:800])

### Prometheus + Grafana (Docker)

Jalankan di terminal **lokal**:

**Prometheus**
```bash
docker run -p 9090:9090 \
  -v "$(pwd)/Monitoring_dan_Logging/2.prometheus.yml:/etc/prometheus/prometheus.yml" \
  prom/prometheus
```

**Grafana**
```bash
docker run -d -p 3000:3000 --name grafana grafana/grafana
```

📸 Screenshot:
- `4.bukti monitoring Prometheus/` (5 metrik)
- `5.bukti monitoring Grafana/` (5 panel)
- `6.bukti alerting Grafana/` (rules + notifikasi)

In [ ]:
# @title Simpan hasil ke Drive (opsional)
from google.colab import drive
drive.mount('/content/drive')

!cp -r {PROJECT_DIR}/Membangun_model/mlruns /content/drive/MyDrive/Dicoding-Training_Model/Membangun_model/
!cp -r {PROJECT_DIR}/Membangun_model/model_output /content/drive/MyDrive/Dicoding-Training_Model/Membangun_model/ 2>/dev/null || true
print("Hasil training disimpan ke Drive")

In [ ]:
# @title Matikan server
serve_proc.kill()
exporter_proc.kill()
print("Server dihentikan.")